In [ ]:
# Chargement et analyses exploratoires du jeu de données d'arbres
# Ce script effectue :
# - chargement et nettoyage de base
# - EDA (têtes, types, valeurs manquantes, top espèces)
# - conversions numériques de variables clés
# - tests statistiques et régressions pour tester les hypothèses proposées
# - tracés simples (matplotlib, un graphique par figure)
#
# Le notebook affichera les résultats automatiquement.
# Si le fichier n'est pas trouvé, vérifier le chemin /mnt/data/arbolado-publico-lineal-2017-2018.csv

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.api as sm
import statsmodels.formula.api as smf
from scipy import stats
import os

# utilitaire d'affichage fourni par l'environnement
try:
    from caas_jupyter_tools import display_dataframe_to_user
except Exception as e:
    display_dataframe_to_user = None

# --- 1) Chargement
path = "../data/arbolado-publico-lineal-2017-2018.csv"


df = pd.read_csv(path, dtype=str, low_memory=False)

# Aperçu initial
print("Dimensions :", df.shape)
print("\nColonnes et types initiaux :")
print(df.dtypes)

# afficher un extrait interactif si la fonction est disponible
if display_dataframe_to_user is not None:
    display_dataframe_to_user("Aperçu du jeu de données (raw)", df.head(200))
else:
    print("\nExtrait (5 premières lignes) :")
    display(df.head())

# --- 2) Nettoyage et conversion
# Colonnes numériques attendues : long, lat, diametro_altura_pecho, altura_arbol, ancho_acera
def to_numeric_col(s):
    if s is None:
        return np.nan
    # remplacer virgules décimales par points, supprimer espaces
    return pd.to_numeric(s.str.replace(",", ".").str.strip(), errors="coerce")

df_clean = df.copy()

# standardisation des noms de colonnes (minuscules, sans espaces)
df_clean.columns = [c.strip() for c in df_clean.columns]

# conversions
for col in ["long", "lat", "diametro_altura_pecho", "altura_arbol", "ancho_acera"]:
    if col in df_clean.columns:
        df_clean[col + "_num"] = to_numeric_col(df_clean[col].astype(str))
    else:
        print(f"Avertissement : colonne {col} non trouvée.")

# Extraire top espèces et créer variable d'espèce réduite pour tests
if "nombre_cientifico" in df_clean.columns:
    df_clean["nombre_cientifico"] = df_clean["nombre_cientifico"].fillna("Desconocida").str.strip()
    top_species = df_clean["nombre_cientifico"].value_counts().nlargest(20)
    print("\nTop 20 espèces :\n", top_species)
else:
    print("Colonne 'nombre_cientifico' absente.")

# --- 3) Hypothèse H2 : corrélation diamètre vs hauteur
print("\n--- Test H2 : corrélation diamètre (cm) vs hauteur (m) ---")
dia = df_clean.get("diametro_altura_pecho_num")
alt = df_clean.get("altura_arbol_num")
mask = dia.notna() & alt.notna()
print("Nombre paires valides :", mask.sum())

if mask.sum() >= 10:
    r, p = stats.pearsonr(dia[mask], alt[mask])
    print(f"Corrélation de Pearson r = {r:.3f}, p-value = {p:.3e}")
    # scatter plot
    plt.figure(figsize=(6,4))
    plt.scatter(dia[mask], alt[mask], s=8)
    plt.xlabel("Diamètre à la hauteur de la poitrine (cm)")
    plt.ylabel("Hauteur de l'arbre (m)")
    plt.title("Diamètre vs Hauteur")
    plt.grid(True)
    plt.show()
else:
    print("Données insuffisantes pour test de corrélation.")

# --- 4) Hypothèse H6 : hauteur ~ ancho_acera (régression linéaire)
print("\n--- Test H6 : régression hauteur ~ ancho_acera ---")
ancho = df_clean.get("ancho_acera_num")
mask2 = alt.notna() & ancho.notna()
print("Nombre paires valides :", mask2.sum())

if mask2.sum() >= 20:
    X = sm.add_constant(ancho[mask2])
    model = sm.OLS(alt[mask2], X, missing='drop').fit()
    print(model.summary())
    # tracer
    plt.figure(figsize=(6,4))
    plt.scatter(ancho[mask2], alt[mask2], s=8)
    # droite de régression
    xx = np.linspace(ancho[mask2].min(), ancho[mask2].max(), 100)
    yy = model.params["const"] + model.params[ancho.name] * xx
    plt.plot(xx, yy)
    plt.xlabel("Largeur du trottoir (m)")
    plt.ylabel("Hauteur de l'arbre (m)")
    plt.title("Hauteur en fonction de la largeur du trottoir")
    plt.grid(True)
    plt.show()
else:
    print("Données insuffisantes pour régression hauteur ~ ancho_acera.")

# --- 5) Hypothèse H1 : diamètre moyen par comuna (ANOVA)
print("\n--- Test H1 : diamètre moyen selon la comuna (ANOVA) ---")
if "comuna" in df_clean.columns and "diametro_altura_pecho_num" in df_clean.columns:
    # prendre communes avec au moins 30 observations pour robustesse
    grouped = df_clean[df_clean["diametro_altura_pecho_num"].notna()].groupby("comuna")["diametro_altura_pecho_num"].agg(["count","mean"]).sort_values("count", ascending=False)
    print("\nObservations par comuna (top 10) :\n", grouped.head(10))
    selected_comunas = grouped[grouped["count"] >= 30].index.tolist()
    print("\nCommunes retenues (>=30 obs) :", selected_comunas[:10])
    samples = [df_clean.loc[(df_clean["comuna"]==c) & df_clean["diametro_altura_pecho_num"].notna(), "diametro_altura_pecho_num"].values for c in selected_comunas]
    if len(samples) >= 2:
        F, p_anova = stats.f_oneway(*samples)
        print(f"ANOVA one-way: F = {F:.3f}, p = {p_anova:.3e}")
    else:
        print("Pas assez de groupes avec >=30 observations pour ANOVA.")
else:
    print("Colonnes 'comuna' ou 'diametro_altura_pecho_num' manquantes.")

# --- 6) Hypothèse H4 : distribution des espèces par comuna (chi2)
print("\n--- Test H4 : espèces vs comuna (chi2) sur top espèces) ---")
if "nombre_cientifico" in df_clean.columns and "comuna" in df_clean.columns:
    # construire table croisée avec top N espèces
    topN = 10
    topN_species = df_clean["nombre_cientifico"].value_counts().nlargest(topN).index.tolist()
    ctab = pd.crosstab(df_clean.loc[df_clean["nombre_cientifico"].isin(topN_species), "nombre_cientifico"],
                       df_clean.loc[df_clean["nombre_cientifico"].isin(topN_species), "comuna"])
    print("Table croisée (espèces x comuna) : dimensions", ctab.shape)
    if ctab.size > 0:
        chi2, p_chi, dof, expected = stats.chi2_contingency(ctab)
        print(f"Chi2 = {chi2:.2f}, p-value = {p_chi:.3e}, dof = {dof}")
    else:
        print("Table croisée vide - vérifier valeurs.")
    if display_dataframe_to_user is not None:
        display_dataframe_to_user("Table croisée top espèces x comuna", ctab)
    else:
        display(ctab)
else:
    print("Colonnes 'nombre_cientifico' ou 'comuna' manquantes.")

# --- 7) Hypothèse H8/H9 : association état_plantera vs ubicacion_plantera (test de contingence)
print("\n--- Test H8 : estado_plantera vs ubicacion_plantera (chi2) ---")
if "estado_plantera" in df_clean.columns and "ubicacion_plantera" in df_clean.columns:
    ctab2 = pd.crosstab(df_clean["estado_plantera"].fillna("NA"), df_clean["ubicacion_plantera"].fillna("NA"))
    if ctab2.size > 0:
        chi2_2, p2, dof2, expected2 = stats.chi2_contingency(ctab2)
        print(f"Chi2 = {chi2_2:.2f}, p-value = {p2:.3e}, dof = {dof2}")
        if display_dataframe_to_user is not None:
            display_dataframe_to_user("estado_plantera x ubicacion_plantera", ctab2)
        else:
            display(ctab2)
    else:
        print("Table vide")
else:
    print("Colonnes 'estado_plantera' ou 'ubicacion_plantera' manquantes.")

# --- 8) Visualisation spatiale simple : scatter lat/long colorée par hauteur
print("\n--- Visualisation spatiale : lat/long (taille point ~ hauteur) ---")
if "long_num" in df_clean.columns and "lat_num" in df_clean.columns and "altura_arbol_num" in df_clean.columns:
    mask_sp = df_clean["long_num"].notna() & df_clean["lat_num"].notna()
    if mask_sp.sum() > 0:
        plt.figure(figsize=(6,6))
        sizes = np.clip(df_clean.loc[mask_sp, "altura_arbol_num"].fillna(0).values, 0, 30)
        plt.scatter(df_clean.loc[mask_sp, "long_num"], df_clean.loc[mask_sp, "lat_num"], s= (sizes+1)*2, alpha=0.6)
        plt.xlabel("Longitude")
        plt.ylabel("Latitude")
        plt.title("Position des arbres (taille point ~ hauteur)")
        plt.grid(True)
        plt.show()
    else:
        print("Pas de coordonnées valides pour tracer la carte.")
else:
    print("Colonnes long/lat/altura manquantes pour la visualisation spatiale.")

# --- 9) Synthèse rapide des résultats (affichée)
print("\n--- Synthèse rapide ---")
synth = []
if mask.sum() >= 10:
    synth.append(f"Corrélation diamètre-hauteur: r={r:.3f}, p={p:.3e}")
if mask2.sum() >= 20:
    synth.append(f"Régression hauteur~ancho_acera: coef largeur={model.params[ancho.name]:.4f}, p={model.pvalues[ancho.name]:.3e}")
if 'p_anova' in locals():
    synth.append(f"ANOVA diamètre~comuna: F={F:.3f}, p={p_anova:.3e}")
if 'p_chi' in locals():
    synth.append(f"Chi2 espèces x comuna: p={p_chi:.3e}")
if 'p2' in locals():
    synth.append(f"Chi2 estado_plantera x ubicacion_plantera: p={p2:.3e}")

for line in synth:
    print("- " + line)

# Fin du script. Tu peux modifier ce notebook pour approfondir chaque test.



FileNotFoundError: [Errno 2] No such file or directory: './data/arbolado-publico-lineal-2017-2018.csv'